In [ ]:
# | default_exp transforms/monai/croppad

# Imports

In [ ]:
# | export


from collections.abc import Hashable, Mapping, Sequence
from typing import Any

import torch
from monai.config import KeysCollection
from monai.data import MetaTensor, get_track_meta
from monai.transforms import CropForegroundd, MapTransform, RandSpatialCropSamples, RandSpatialCropSamplesd, SpatialCrop
from monai.utils import ImageMetaKey as Key

In [ ]:
import numpy as np

# Utilty functions

In [ ]:
# | export


def get_updated_crop_start(current_crop_start, new_crop_start):
    if not torch.is_tensor(new_crop_start):
        new_crop_start = torch.tensor(new_crop_start)

    if current_crop_start is None:
        return new_crop_start

    if not torch.is_tensor(current_crop_start):
        current_crop_start = torch.tensor(current_crop_start)

    updated_crop_start = current_crop_start + new_crop_start
    return updated_crop_start

In [ ]:
# Original image size: 256x256
# First time cropping to size 100x101 with starting at (50, 60)


original_size = np.array((256, 256))

current_crop_start = None
new_crop_start = np.array((50, 60))

updated_crop_start = get_updated_crop_start(current_crop_start, new_crop_start)
updated_crop_start

tensor([50, 60])

In [ ]:
# Now cropping that new image to size 50x50 with starting at (10, 11)

current_crop_start = updated_crop_start
new_crop_start = np.array((10, 11))

updated_crop_start = get_updated_crop_start(current_crop_start, new_crop_start)
updated_crop_start

tensor([60, 71])

# Transforms

In [ ]:
# | export


class CropForegroundWithCropTrackingd(CropForegroundd):
    def __init__(
        self,
        keys,
        crop_offset_key: str = "crop_offset",
        *args,
        **kwargs,
    ) -> MetaTensor:
        super().__init__(keys, *args, **kwargs)
        self.crop_offset_key = crop_offset_key

    def __call__(self, data, *args, **kwargs):
        output = super().__call__(data, *args, **kwargs)
        crop_offset = output[self.start_coord_key]
        output[self.crop_offset_key] = get_updated_crop_start(output.get(self.crop_offset_key), crop_offset)
        return output

In [ ]:
img = torch.zeros(4, 100, 100, 100)
img[0, 20:30, 30:40, 40:50] = 1
img[1, 10:20, 30:40, 40:50] = 1
img[2, 20:30, 20:30, 40:50] = 1
img[3, 20:30, 30:40, 50:60] = 1
x = {"image": img, "crop_offset": (1, 1, 1)}

CropForegroundWithCropTrackingd(keys=["image"], source_key="image")(x)


{
    'image': metatensor([[[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]],

         [[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]],

         [[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]],

         ...,

         [[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [1., 1.

In [ ]:
# | export


class RandSpatialCropSamplesWithCropTracking(RandSpatialCropSamples):  # To return the crops along with the crop offset
    def __init__(self, crop_offset_key: str = "crop_offset", *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.crop_offset_key = crop_offset_key

    def __call__(self, img: torch.Tensor, lazy: bool | None = None) -> list[torch.Tensor]:
        """
        Apply the transform to `img`, assuming `img` is channel-first and
        cropping doesn't change the channel dim.
        """
        ret = []
        lazy_ = self.lazy if lazy is None else lazy
        for i in range(self.num_samples):
            cropped = self.cropper(img, lazy=lazy_)
            if get_track_meta():
                cropped.meta[self.crop_offset_key] = tuple(_slice.start for _slice in self.cropper._slices)
                cropped.meta[Key.PATCH_INDEX] = i  # type: ignore
                self.push_transform(cropped, replace=True, lazy=lazy_)  # track as this class instead of RandSpatialCrop
            ret.append(cropped)
        return ret


class RandSpatialCropSamplesWithCropTrackingd(RandSpatialCropSamplesd):
    def __init__(
        self,
        keys,
        roi_size: Sequence[int] | int,
        num_samples: int,
        max_roi_size: Sequence[int] | int | None = None,
        random_center: bool = True,
        random_size: bool = False,
        allow_missing_keys: bool = False,
        lazy: bool = False,
        crop_offset_key: str = "crop_offset",
    ) -> MetaTensor:
        super().__init__(
            keys, roi_size, num_samples, max_roi_size, random_center, random_size, allow_missing_keys, lazy
        )
        self.crop_offset_key = crop_offset_key
        self.cropper = RandSpatialCropSamplesWithCropTracking(
            crop_offset_key, roi_size, num_samples, max_roi_size, random_center, random_size, lazy=lazy
        )

    def __call__(self, data, *args, **kwargs):
        output = super().__call__(data, *args, **kwargs)
        for key in self.keys:
            for o in output:
                crop_offset = o[key].meta.get(self.crop_offset_key)
                o[self.crop_offset_key] = get_updated_crop_start(o.get(self.crop_offset_key), crop_offset)
        return output

In [ ]:
x = {"image": img, "crop_offset": (1, 1, 1)}

RandSpatialCropSamplesWithCropTrackingd(
    keys="image",
    roi_size=(50, 50, 50),
    max_roi_size=(60, 60, 60),
    num_samples=2,
    random_size=True,
)(x)[0]


{
    'image': metatensor([[[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]],

         [[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]],

         [[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]],

         ...,

         [[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0.

In [ ]:
# | export


class BBoxCenterCropd(MapTransform):
    """Crop a fixed-size ROI centered on the midpoint of a bounding box.

    When the centered window would extend beyond the image boundary, it is
    shifted inward so that the full ``roi_size`` is always returned (assuming
    the spatial dimensions of the image are >= ``roi_size``).  This matches the
    behaviour of the ``_extract_centered_crop`` helper used in the legacy
    dataloader.

    The bounding box is read from ``data[bbox_key]`` and is expected to be a
    sequence of ``(z1, y1, x1, z2, y2, x2)`` **in voxel coordinates**.

    Args:
        keys: keys of the image tensors to crop (channel-first, i.e.
            ``(C, D, H, W)``).
        bbox_key: dictionary key that holds the bounding box.
        roi_size: desired spatial size ``(D, H, W)`` of the crop.
        allow_missing_keys: if ``True``, do not raise if a key in *keys* is
            absent from the data dict.
    """

    def __init__(
        self,
        keys: KeysCollection,
        bbox_key: str = "bbox",
        roi_size: Sequence[int] | list[int] | tuple[int, ...] | None = None,
        allow_missing_keys: bool = False,
    ) -> None:
        super().__init__(keys, allow_missing_keys=allow_missing_keys)
        self.bbox_key = bbox_key
        self.roi_size = tuple(roi_size)

    @staticmethod
    def _compute_crop_slices(
        bbox: Sequence[int],
        roi_size: tuple[int, ...],
        spatial_shape: tuple[int, ...],
    ) -> tuple[list[slice], list[int]]:
        """Return ``(slices, start_coords)`` for the crop.

        ``slices`` can be fed directly to :class:`SpatialCrop` (one per
        spatial dim, excluding the channel dim).
        """
        z1, y1, x1, z2, y2, x2 = (int(b) for b in bbox)
        centers = ((z1 + z2) // 2, (y1 + y2) // 2, (x1 + x2) // 2)

        starts: list[int] = []
        ends: list[int] = []
        for center, size, dim_len in zip(centers, roi_size, spatial_shape):
            half = size // 2
            s = center - half
            e = s + size

            # Shift window inward when it overflows
            if s < 0:
                s, e = 0, size
            if e > dim_len:
                e = dim_len
                s = e - size
            # Guard against image smaller than roi_size
            s = max(s, 0)

            starts.append(s)
            ends.append(e)

        slices = [slice(s, e) for s, e in zip(starts, ends)]
        return slices, starts

    def __call__(self, data: Mapping[Hashable, Any]) -> dict[Hashable, Any]:
        d = dict(data)
        bbox = d[self.bbox_key]

        # We need the spatial shape from the first available key to compute
        # boundary-aware start/end.  Spatial dims are everything after channel.
        first_key = next(k for k in self.key_iterator(d))
        spatial_shape = d[first_key].shape[1:]  # (D, H, W)

        slices, _ = self._compute_crop_slices(bbox, self.roi_size, spatial_shape)
        cropper = SpatialCrop(roi_slices=slices)

        for key in self.key_iterator(d):
            d[key] = cropper(d[key])

        return d


class BBoxCenterCropWithCropTrackingd(BBoxCenterCropd):
    """``BBoxCenterCropd`` with crop-offset tracking.

    Accumulates the crop start coordinate into ``data[crop_offset_key]`` so
    that downstream transforms (or post-processing) can map coordinates back
    to the original volume.

    Args:
        keys: keys of the image tensors to crop.
        bbox_key: dictionary key that holds the bounding box.
        roi_size: desired spatial size ``(D, H, W)`` of the crop.
        crop_offset_key: key under which to store / accumulate the crop
            start coordinate.
        allow_missing_keys: if ``True``, do not raise if a key in *keys* is
            absent from the data dict.
    """

    def __init__(
        self,
        keys: KeysCollection,
        bbox_key: str = "bbox",
        roi_size: Sequence[int] | list[int] | tuple[int, ...] | None = None,
        crop_offset_key: str = "crop_offset",
        allow_missing_keys: bool = False,
    ) -> None:
        super().__init__(keys, bbox_key=bbox_key, roi_size=roi_size, allow_missing_keys=allow_missing_keys)
        self.crop_offset_key = crop_offset_key

    def __call__(self, data: Mapping[Hashable, Any]) -> dict[Hashable, Any]:
        d = dict(data)
        bbox = d[self.bbox_key]

        first_key = next(k for k in self.key_iterator(d))
        spatial_shape = d[first_key].shape[1:]

        slices, starts = self._compute_crop_slices(bbox, self.roi_size, spatial_shape)
        cropper = SpatialCrop(roi_slices=slices)

        for key in self.key_iterator(d):
            d[key] = cropper(d[key])

        d[self.crop_offset_key] = get_updated_crop_start(d.get(self.crop_offset_key), starts)

        return d

In [ ]:
# --- Test BBoxCenterCropd (no tracking) ---


img = torch.arange(1 * 100 * 100 * 100, dtype=torch.float32).reshape(1, 100, 100, 100)
bbox_mid = [40, 45, 50, 60, 55, 60]  # center = (50, 50, 55)

out = BBoxCenterCropd(keys="image", bbox_key="bbox", roi_size=(64, 64, 64))({"image": img, "bbox": bbox_mid})
assert out["image"].shape == (1, 64, 64, 64)
assert "crop_offset" not in out, "BBoxCenterCropd should not write crop_offset"
expected_crop = img[:, 18:82, 18:82, 23:87]
assert torch.equal(out["image"], expected_crop)
print("[PASS] BBoxCenterCropd: basic crop, no tracking, correct voxels")

# --- Test BBoxCenterCropWithCropTrackingd ---

# 1) Basic center crop
out = BBoxCenterCropWithCropTrackingd(keys="image", bbox_key="bbox", roi_size=(64, 64, 64))(
    {"image": img, "bbox": bbox_mid}
)
assert out["image"].shape == (1, 64, 64, 64)
assert list(out["crop_offset"]) == [18, 18, 23]
print(f"[PASS] Center crop: shape={tuple(out['image'].shape)}, offset={out['crop_offset'].tolist()}")

# 2) Boundary shift: bbox near z=0 edge → crop should shift to start at z=0
bbox_low = [2, 50, 50, 8, 55, 55]  # center_z = 5, half = 32 → would start at -27
out2 = BBoxCenterCropWithCropTrackingd(keys="image", bbox_key="bbox", roi_size=(64, 64, 64))(
    {"image": img, "bbox": bbox_low}
)
assert out2["image"].shape == (1, 64, 64, 64)
assert out2["crop_offset"][0].item() == 0
print(f"[PASS] Low-edge shift: offset={out2['crop_offset'].tolist()}")

# 3) Boundary shift: bbox near z=max edge → crop should shift to end at 100
bbox_high = [92, 50, 50, 98, 55, 55]  # center_z = 95, half=32 → would end at 127
out3 = BBoxCenterCropWithCropTrackingd(keys="image", bbox_key="bbox", roi_size=(64, 64, 64))(
    {"image": img, "bbox": bbox_high}
)
assert out3["image"].shape == (1, 64, 64, 64)
assert out3["crop_offset"][0].item() == 36
print(f"[PASS] High-edge shift: offset={out3['crop_offset'].tolist()}")

# 4) Crop offset accumulation: pre-existing offset should be added
out4 = BBoxCenterCropWithCropTrackingd(keys="image", bbox_key="bbox", roi_size=(64, 64, 64))(
    {"image": img, "bbox": bbox_mid, "crop_offset": torch.tensor([10, 20, 30])}
)
assert list(out4["crop_offset"]) == [28, 38, 53]
print(f"[PASS] Offset accumulation: {out4['crop_offset'].tolist()}")

# 5) Multiple keys: both "image" and "mask" are cropped identically
mask = torch.ones(1, 100, 100, 100, dtype=torch.float32)
out5 = BBoxCenterCropWithCropTrackingd(keys=["image", "mask"], bbox_key="bbox", roi_size=(64, 64, 64))(
    {"image": img, "mask": mask, "bbox": bbox_mid}
)
assert out5["image"].shape == out5["mask"].shape == (1, 64, 64, 64)
print("[PASS] Multi-key crop: both (1,64,64,64)")

# 6) Verify voxel values match manual slice
assert torch.equal(out["image"], expected_crop)
print("[PASS] Voxel values match manual slice")

print("\nAll BBoxCenterCrop tests passed!")

[PASS] BBoxCenterCropd: basic crop, no tracking, correct voxels
[PASS] Center crop: shape=(1, 64, 64, 64), offset=[18, 18, 23]
[PASS] Low-edge shift: offset=[0, 20, 20]
[PASS] High-edge shift: offset=[36, 20, 20]
[PASS] Offset accumulation: [28, 38, 53]
[PASS] Multi-key crop: both (1,64,64,64)
[PASS] Voxel values match manual slice

All BBoxCenterCrop tests passed!


# nbdev

In [ ]:
!nbdev_export